In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


#### Busca Dados IPCA via código Atualiza-Indicador

In [3]:

from database.conexao import conectar
from indicadores.ipca import Ipca

with conectar():
    dados_ipca = Ipca.buscar()
    for aa in dados_ipca:
        print(aa)


{'data': '01/09/2025', 'valor': '0.48'}
{'data': '01/10/2025', 'valor': '0.09'}
{'data': '01/11/2025', 'valor': '0.18'}
{'data': '01/12/2025', 'valor': '0.33'}
{'data': '01/01/2026', 'valor': '0.33'}
{'data': '01/02/2026', 'valor': '0.70'}
{'data': '01/03/2026', 'valor': '0.88'}
{'data': '01/04/2026', 'valor': '0.67'}
{'data': '01/05/2026', 'valor': '0.58'}
{'data': '01/06/2026', 'valor': '0.16'}
{'data': '01/07/2026', 'valor': '0.07'}
{'data': '01/08/2026', 'valor': '-0.32'}


#### Busca Dados IPCA via código Atualiza-Indicador

In [4]:

from database.conexao import conectar
from indicadores.ipca import Ipca

with conectar():
    dados_ipca = Ipca.select()
    for aa in dados_ipca:
        print(aa.indice)
        print(aa.dt_referencia)


0.48000
2025-09-01
0.09000
2025-10-01
0.18000
2025-11-01
0.33000
2025-12-01
0.33000
2026-01-01
0.70000
2026-02-01
0.88000
2026-03-01
0.67000
2026-04-01
0.58000
2026-05-01
0.16000
2026-06-01
0.07000
2026-07-01
-0.32000
2026-08-01


#### Busca Dados IPCA Diretamente da API

In [5]:

ultimos_meses = 5

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados/ultimos/{ultimos_meses}?formato=json"

#parametros = { "formato": "json", "dataInicial": data_inicial, "dataFinal": data_final }

resposta = requests.get(url, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for ipca in dados:
    print(ipca)


{'data': '01/04/2026', 'valor': '0.67'}
{'data': '01/05/2026', 'valor': '0.58'}
{'data': '01/06/2026', 'valor': '0.16'}
{'data': '01/07/2026', 'valor': '0.07'}
{'data': '01/08/2026', 'valor': '-0.32'}


#### Gera datas

In [6]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=230)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


Período: 04/02/2026 até 22/09/2026


#### Busca Dados IPCA da API com parêmetro de datas

In [7]:

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Percorrer a lista de dicionário
for ipca in dados:
    print(ipca)


{'data': '01/02/2026', 'valor': '0.70'}
{'data': '01/03/2026', 'valor': '0.88'}
{'data': '01/04/2026', 'valor': '0.67'}
{'data': '01/05/2026', 'valor': '0.58'}
{'data': '01/06/2026', 'valor': '0.16'}
{'data': '01/07/2026', 'valor': '0.07'}
{'data': '01/08/2026', 'valor': '-0.32'}


#### Insere Dados no Banco (transformando dados)

In [8]:
# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Transforma dados para o formato da tabela Igpm
dados_ipca = []
for registro in dados:
    dados_ipca.append({"indice": registro["valor"],
                       "status": True,
                       "dt_referencia": datetime.strptime(registro["data"], "%d/%m/%Y").date()
                      })

with conectar():
    Ipca.insert_many(dados_ipca).on_conflict_ignore().execute()

#### Visualiza DADOS IPCA (lista de dicionário)

In [9]:
for ipca in dados:
    print(ipca["data"], ipca["valor"])


01/02/2026 0.70
01/03/2026 0.88
01/04/2026 0.67
01/05/2026 0.58
01/06/2026 0.16
01/07/2026 0.07
01/08/2026 -0.32


#### Carga direta no Banco (registro por registro)

In [ ]:

from database.conexao import conectar
from indicadores.ipca import Ipca
from decimal import Decimal

with conectar():
    for registro in dados:
        Ipca.insert(indice=registro["valor"], status=True, dt_referencia=registro["data"]).on_conflict_ignore().execute()


#### Carga direta no Banco (Em Massa)

In [ ]:

from database.conexao import conectar
from indicadores.ipca import Ipca
from decimal import Decimal

with conectar():
    Ipca.insert_many(dados_ipca).on_conflict_ignore().execute()

#### Atualiza tabela IPCA usando método do código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.ipca import Ipca

aa = Ipca.atualizar_ipca()

print(aa)